<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/22modelDropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dropout:

During training → randomly turns off neurons
Like studying without some brain cells each time
Forces other neurons to learn
Result → less overfitting!

Batch Normalization:

Normalizes output of each layer
Like scaling features but inside the network
Makes training faster and more stable
Helps gradients flow better
📌 Build 3 Models & Compare

Model A — No Dropout, No BatchNorm (baseline):

Linear(8, 64) → ReLU
Linear(64, 32) → ReLU
Linear(32, 1) → Sigmoid

Model B — With Dropout:

Linear(8, 64) → ReLU → Dropout(0.3)
Linear(64, 32) → ReLU → Dropout(0.3)
Linear(32, 1) → Sigmoid

Model C — Dropout + BatchNorm:

Linear(8, 64) → BatchNorm1d(64) → ReLU → Dropout(0.3)
Linear(64, 32) → BatchNorm1d(32) → ReLU → Dropout(0.3)
Linear(32, 1) → Sigmoid
📌 Training Each Model
Loss → BCELoss
Optimizer → Adam lr=0.001
Epochs → 1000
Print loss every 100 epochs

Important — For Dropout:

During training → model.train()
During evaluation → model.eval()
This turns dropout OFF during eval automatically!
📌 Must Do — Compare Train vs Test Accuracy

For each model print:

Training accuracy
Test accuracy
Gap between them
Model	Train Acc	Test Acc	Gap
No Dropout	?	?	?
Dropout 0.3	?	?	?
BatchNorm + Dropout	?	?	?

Smaller gap = less overfitting = better model!

🧠 Understand This
Why dropout only active during training?
What does model.train() vs model.eval() do?
Why does BatchNorm help?
What happens with dropout=0.5 vs 0.3?

In [ ]:
import torch

In [ ]:
import numpy as np
import pandas as pp

tt = pp.read_csv('/content/tatnic.csv')

In [ ]:
tt.isnull().sum()

In [ ]:
x = tt['Age'].mean()
tt['Age'] = tt['Age'].replace(np.nan,x)

In [ ]:
x= tt['Fare'].mean()
tt['Fare']= tt['Fare'].replace(np.nan,x)

In [ ]:
X = tt.drop(['Unnamed: 0','Cabin','Survived'],axis=1)

In [ ]:
y = tt['Survived']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [ ]:
X_train = torch.FloatTensor(X_train.values)
X_test = torch.FloatTensor(X_test.values)
y_train = torch.FloatTensor(y_train.values).unsqueeze(1)
y_test = torch.FloatTensor(y_test.values).unsqueeze(1)

In [ ]:
from torch import nn
class tri(nn.Module):
    def __init__(self,input):
        super().__init__()

        self.lin =nn.Sequential(
            nn.Linear(input,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16,1),
            nn.Sigmoid()

        )
    def forward(self,x):
        return self.lin(x)


In [ ]:
model = tri(X_train.shape[1])

loss_fn = nn.BCELoss()

optim = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epos = 150

for epos in range(epos):

    y_pre = model(X_train)

    loss = loss_fn(y_pre,y_train)

    optim.zero_grad()

    loss.backward()

    optim.step()

    print(f'epos {epos}, loss{loss}')

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)

    predicted = (outputs > 0.5).float()

    correct = (predicted == y_test).sum().item()
    total = y_test.size(0)

    accuracy = correct / total * 100

print("Final Accuracy:", accuracy, "%")